<h1 style=\"text-align: center; font-size: 50px;\"> 🤖 MLFlow Registration for Agentic RAG Model</h1>

# Notebook Overview

- Start Execution
- Install and Import Libraries
- Configure Settings
- Define the Agentic RAG Model
- Register the Model to MLFlow
- Log Results to MLFlow

# Start Execution

In [1]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [2]:
start_time = time.time()  
logger.info("Notebook execution started.")

2025-07-24 17:08:49 - INFO - Notebook execution started.


# Install and Import Libraries

In [3]:
%%time

%pip install -r ../requirements.txt --quiet 

Note: you may need to restart the kernel to use updated packages.
CPU times: user 89.7 ms, sys: 21.5 ms, total: 111 ms
Wall time: 2.72 s


In [ ]:
from __future__ import annotations

import json
import os
import sys
import warnings
from collections import namedtuple
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, TypedDict

import pandas as pd
import tensorrt_llm
from tensorrt_llm.llms import TensorRTLangchain

import mlflow.pyfunc
from mlflow.models.signature import ModelSignature
from mlflow.tracking import MlflowClient
from mlflow.types import ColSpec, DataType, Schema

from langchain.docstore.document import Document
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langgraph.graph import StateGraph, START, END

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[TensorRT-LLM] TensorRT-LLM version: 0.18.0


# Configure Settings

In [5]:
# ------------------------ Suppress Verbose Logs ------------------------
warnings.filterwarnings("ignore")

In [6]:
# ------------------------- MLflow Experiment Configuration -------------------------
MODEL_NAME = "Agentic_RAG_Model"
RUN_NAME = f"Register_{MODEL_NAME}_Run"
EXPERIMENT_NAME = "Agentic_RAG_Experiment"

# Define the Agentic RAG Model

In [ ]:
        logger.info(f"✅ RagAgenticModel logged with multiple components:")
        logger.info("   📦 ONNX: sentence_transformer_embedding.onnx (CPU-optimized embeddings)")
        logger.info("   📦 ONNX: tensorrt_llm_nemotron.onnx (Exported TensorRT LLM)")
        logger.info("   🗃️ Artifacts: Chroma vectorstore + memory system")

/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:168: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


# Register the Model to MLFlow

In [8]:
# 1. Set MLflow tracking URI and experiment
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "/phoenix/mlflow"))
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)
print(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

2025/07/24 17:09:03 INFO mlflow.tracking.fluent: Experiment with name 'Agentic_RAG_Experiment' does not exist. Creating a new experiment.


Using MLflow tracking URI: /phoenix/mlflow
Experiment: Agentic_RAG_Experiment


In [9]:
%%time

# 2. Start an MLflow run and log + register the model
with mlflow.start_run(run_name=RUN_NAME) as run:
    print(f"Started MLflow run: {run.info.run_id}")

    # Log RagAgenticModel using the class method
    RagAgenticModel.log_model(model_name=MODEL_NAME)

    model_uri = f"runs:/{run.info.run_id}/{MODEL_NAME}"
    mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)

# ------------------------- Success Confirmation -------------------------

print(f"✅ Model '{MODEL_NAME}' successfully logged and registered under experiment '{EXPERIMENT_NAME}'.")

Started MLflow run: ef738130b1d14e1ea32c19a5a47a1042


2025-07-24 17:09:08,702 [INFO] RagAgenticModel.log_model: Logged RagAgenticModel under artifact_path 'Agentic_RAG_Model'
Successfully registered model 'Agentic_RAG_Model'.


✅ Model 'Agentic_RAG_Model' successfully logged and registered under experiment 'Agentic_RAG_Experiment'.
CPU times: user 2.16 s, sys: 681 ms, total: 2.84 s
Wall time: 5.8 s


Created version '1' of model 'Agentic_RAG_Model'.


In [10]:
# 3. Retrieve the latest version from the Model Registry
client = MlflowClient()
versions = client.get_latest_versions(MODEL_NAME, stages=["None"])
if not versions:
    raise RuntimeError(f"No registered versions found for model '{MODEL_NAME}'.")
latest_version = versions[0].version

model_info = mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{latest_version}")
print(f"Latest registered version of '{MODEL_NAME}': {latest_version}")
print(f"Signature: {model_info.signature}")

Latest registered version of 'Agentic_RAG_Model': 1
Signature: inputs: 
  ['query': string (required)]
outputs: 
  None
params: 
  None



# Log Results to MLFlow

In [11]:
%%time

# 4. Load the model from the Model Registry
loaded_model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{latest_version}")
print(f"Successfully loaded model '{MODEL_NAME}' version {latest_version} for inference.")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Loading Model: [1/3]	Downloading HF model
Downloaded model to /root/.cache/huggingface/hub/models--nvidia--Llama-3.1-Nemotron-Nano-8B-v1/snapshots/a22e1c57330633cd3522903f9bb82480bf3192a6
Time: 0.493s
Loading Model: [2/3]	Loading HF model to memory
230it [00:00, 432.31it/s]
Time: 1.091s
Loading Model: [3/3]	Building TRT-LLM engine
Time: 900.574s
Loading model done.
Total latency: 902.170s


[TensorRT-LLM] TensorRT-LLM version: 0.18.0
[TensorRT-LLM][INFO] Engine version 0.18.0 found in the config file, assuming engine(s) built by new builder API.
[TensorRT-LLM][INFO] Refreshed the MPI local session
[TensorRT-LLM][INFO] MPI size: 1, MPI local size: 1, rank: 0
[TensorRT-LLM][INFO] Rank 0 is using GPU 0
[TensorRT-LLM][WARNING] Fix optionalParams : KV cache reuse disabled because model was not built with paged context FMHA support
[TensorRT-LLM][INFO] TRTGptModel maxNumSequences: 2048
[TensorRT-LLM][INFO] TRTGptModel maxBatchSize: 2048
[TensorRT-LLM][INFO] TRTGptModel maxBeamWidth: 1
[TensorRT-LLM][INFO] TRTGptModel maxSequenceLen: 131072
[TensorRT-LLM][INFO] TRTGptModel maxDraftLen: 0
[TensorRT-LLM][INFO] TRTGptModel mMaxAttentionWindowSize: (131072) * 32
[TensorRT-LLM][INFO] TRTGptModel enableTrtOverlap: 0
[TensorRT-LLM][INFO] TRTGptModel normalizeLogProbs: 0
[TensorRT-LLM][INFO] TRTGptModel maxNumTokens: 8192
[TensorRT-LLM][INFO] TRTGptModel maxInputLen: 8192 = min(maxSeque

In [12]:
# 5. Run a sample inference using the loaded model
sample_query = "What is the hardware requirement for AI Studio?"
input_payload = {"query": sample_query}

print("\n=== Running Sample Inference ===")
result = loaded_model.predict(input_payload)

2025-07-24 17:25:32,473 [INFO] RagAgenticModel: Received user query: What is the hardware requirement for AI Studio?



=== Running Sample Inference ===
MODEL INPUT
<class 'pandas.core.frame.DataFrame'>
                                             query
0  What is the hardware requirement for AI Studio?


Processed requests: 100%|██████████| 1/1 [00:30<00:00, 30.60s/it]
2025-07-24 17:26:03,090 [INFO] RagAgenticModel: Relevance check result: True
2025-07-24 17:26:03,092 [INFO] RagAgenticModel: Cache miss for query: What is the hardware requirement for AI Studio?
Processed requests: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]
2025-07-24 17:26:03,903 [INFO] RagAgenticModel: Rewritten query: The required hardware for AI Studio must have at least X GB of RAM and a multi-core processor. (Assuming specific technical details were missing in your note.)
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
2025-07-24 17:26:04,970 [INFO] RagAgenticModel: Retrieved 5 chunks for query.
Processed requests: 100%|██████████| 1/1 [00:00<00:00,  1.36it/s]
2025-07-24 17:26:05,748 [INFO] RagAgenticModel: Generated answer (75 chars)
2025-07-24 17:26:05,757 [INFO] RagAgenticModel: Stored query-answer in memory for key: what is the hardware requirement 

In [13]:
# 6. Print results
print(f"Query:")
print("{sample_query}\n")
print("\n==============\n")

print("Answer:")
print(result.get("answer", "<no answer>"), "\n")
print("\n==============\n")

print("Retrieved Chunks:")
for idx, chunk in enumerate(result.get("retrieved_chunks", []), start=1):
    print(f"  {idx}. {chunk[:100]}{'...' if len(chunk)>100 else ''}")

print("\n==============\n")
print("\nMessage History:")
for msg in result.get("messages", []):
    role = msg.get("role", "<unknown>")
    content = msg.get("content", "")
    print(f"  [{role}]: {content}")

Query:
{sample_query}



Answer:
AMD Ryzen™ 9 processor, Intel Core™ i5 12th generation processor, or higher 



Retrieved Chunks:
  1. Technical Requirements

Hardware:

Windows 10 or 11 or Linux Ubuntu 22.04 LTS on a workstation

GPU ...
  2. Software:

Windows 10 or 11 or Linux Ubuntu 22.04 LTS

Windows OS requires Windows Subsystem for Lin...
  3. title: 'System Requirements' sidebar_position: 1

System Requirements

Z by HP AI Studio currently r...
  4. Distro selection modal

:::tip

If git is not already installed on your machine, the app will guide ...
  5. title: 'Troubleshooting AI Studio' sidebar_position: 6

AI Studio Troubleshooting Guide

Find quick ...



Message History:
  [user]: What is the hardware requirement for AI Studio?
  [developer]: Relevance check result:
  [assistant]: Yes
  [developer]: Rewritten query:
  [assistant]: The required hardware for AI Studio must have at least X GB of RAM and a multi-core processor. (Assuming specific technical details were miss

In [14]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

2025-07-24 17:26:05 - INFO - ⏱️ Total execution time: 17m 16.50s
2025-07-24 17:26:05 - INFO - ✅ Notebook execution completed successfully.


Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).